In [1]:
import pandas as pd
import numpy as np

# Load file train.csv
df = pd.read_csv('train.csv')

# Tampilkan 5 baris pertama
df.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [2]:
# Cek kolom yang memiliki missing value
missing_info = df.isnull().sum()
print("Kolom dengan missing value:")
print(missing_info[missing_info > 0])

# Separasi kolom numerik dan kategorikal
numeric_cols = df.select_dtypes(include=[np.number]).columns
categorical_cols = df.select_dtypes(include=['object']).columns

# Isi missing value kolom numerik dengan median
for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())

# Isi missing value kolom kategorikal dengan modus
for col in categorical_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

print("\nSisa missing value total:", df.isnull().sum().sum())

Kolom dengan missing value:
LotFrontage      259
Alley           1369
MasVnrType       872
MasVnrArea         8
BsmtQual          37
BsmtCond          37
BsmtExposure      38
BsmtFinType1      37
BsmtFinType2      38
Electrical         1
FireplaceQu      690
GarageType        81
GarageYrBlt       81
GarageFinish      81
GarageQual        81
GarageCond        81
PoolQC          1453
Fence           1179
MiscFeature     1406
dtype: int64

Sisa missing value total: 0


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_16412\2017200696.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include=['object']).columns


In [3]:
# Menangani outlier pada target harga rumah (SalePrice) dengan metode IQR
Q1 = df['SalePrice'].quantile(0.25)
Q3 = df['SalePrice'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Filter data tanpa outlier
df_clean = df[(df['SalePrice'] >= lower_bound) & (df['SalePrice'] <= upper_bound)].copy()

print("Jumlah data sebelum hapus outlier:", len(df))
print("Jumlah data setelah hapus outlier:", len(df_clean))

Jumlah data sebelum hapus outlier: 1460
Jumlah data setelah hapus outlier: 1399


In [4]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Fitur numerik untuk di-scale
features_to_scale = ['LotArea', 'GrLivArea', 'GarageArea', 'TotalBsmtSF']

# Transformasi skala
df_clean[features_to_scale] = scaler.fit_transform(df_clean[features_to_scale])

df_clean[features_to_scale].head()

,LotArea,GrLivArea,GarageArea,TotalBsmtSF
0,-0.204916,0.490416,0.436731,-0.413565
1,-0.066726,-0.442178,0.005793,0.584257
2,0.131547,0.648623,0.730553,-0.256272
3,-0.072734,0.504987,0.897052,-0.659333
4,0.493245,1.506277,1.847074,0.296708


In [5]:
from sklearn.model_selection import train_test_split

# Fitur (X) dan Target (y)
X = df_clean[['LotArea', 'GrLivArea', 'GarageArea', 'TotalBsmtSF', 'OverallQual', 'YearBuilt']]
y = df_clean['SalePrice']

# Split data (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Ukuran X_train:", X_train.shape)
print("Ukuran X_test :", X_test.shape)

Ukuran X_train: (1119, 6)
Ukuran X_test : (280, 6)
